# Dependencies

In [16]:
from pathlib import Path
import fitz
from loguru import logger
from tqdm import tqdm
import re

# PDF Parsing

In [5]:
class PDFExtractor:
	def __init__(self, pdf_path: str):
		self.pdf_path = Path(pdf_path)

		if not self.pdf_path.exists():
			raise FileNotFoundError(
				f"PDF file not found: {self.pdf_path}"
			)

	def extract_text(self) -> str:
		"""
		Extract raw text from PDF without any cleaning.
		"""

		logger.info(f"Opening PDF: {self.pdf_path.name}")

		document = fitz.open(self.pdf_path)

		text_parts = []

		for page_num in tqdm(range(len(document)), desc="Extracting pages"):
			page = document.load_page(page_num)

			page_text = page.get_text()

			text_parts.append(
				f"\n\n===== PAGE {page_num + 1} =====\n\n"
			)

			text_parts.append(page_text)
		
		document.close()

		return "".join(text_parts)

	def save_text(self, output_path: str) -> None:
		"""
		Extract and save text into txt file.
		"""

		output_path = Path(output_path)

		output_path.parent.mkdir(parents=True, exist_ok=True)

		text = self.extract_text()

		output_path.write_text(text, encoding="utf-8")

		logger.success(
			f"Text saved to: {output_path}"
		)

In [7]:
if __name__ == "__main__":
	extractor = PDFExtractor(
		pdf_path="regulation/raw/uu/uu_32_2009_pplh.pdf"
	)

	extractor.save_text(
		output_path="regulation/extracted/uu/uu_32_2009_pplh.txt"
	)

2026-06-02 01:16:58.421 | INFO     | __main__:extract_text:15 - Opening PDF: uu_32_2009_pplh.pdf
Extracting pages: 100%|██████████| 110/110 [00:00<00:00, 416.34it/s]
2026-06-02 01:16:58.741 | SUCCESS  | __main__:save_text:49 - Text saved to: regulation\extracted\uu\uu_32_2009_pplh.txt


# Test Schemas

In [1]:
from parser.schemas import Article

In [16]:
from importlib import reload
import parser.schemas
reload(parser.schemas)

<module 'parser.schemas' from 'd:\\1. project adem indonesia\\embedding-model-dev\\parser\\schemas.py'>

In [2]:
article = Article(
    id="uu32_2009_art_63",
    document_type="UU",
    document_number="32",
    document_year=2009,
    document_title="Perlindungan dan Pengelolaan Lingkungan Hidup",
    article_number="63",
    chapter_number="IX",
    chapter_title="Tugas dan Wewenang Pemerintah dan Pemerintah Daerah",
    embedding_text="sample",
    raw_text="sample"
)

print(article.model_dump())

{'id': 'uu32_2009_art_63', 'object_type': 'article', 'document_type': 'UU', 'document_number': '32', 'document_year': 2009, 'document_title': 'Perlindungan dan Pengelolaan Lingkungan Hidup', 'source_pages': [], 'topic_tags': [], 'embedding_text': 'sample', 'raw_text': 'sample', 'article_number': '63', 'chapter_number': 'IX', 'chapter_title': 'Tugas dan Wewenang Pemerintah dan Pemerintah Daerah', 'section_number': None, 'section_title': None, 'paragraphs': []}


# Text Cleaning

In [3]:
from pathlib import Path
import re

from loguru import logger

In [4]:
class TextCleaner:
    def __init__(self):
        pass

    def clean(self, text: str) -> str:
        """
        Apply safe cleaning rules.
        """

        # ---
        # Remove website footer
        # ---
        text = re.sub(
            r"www\.hukumonline\.com",
            "",
            text,
            flags=re.IGNORECASE
        )

        # ---
        # Remove page markers from extractor
        # ---
        text = re.sub(
            r"===== PAGE \d+ =====",
            "",
            text,
        )

        # ---
        # Remove page numbers
        # ---
        text = re.sub(
            r"-\s*\d+\s*-",
            "",
            text,
        )

        # ---
        # Remove common headers
        # PRESIDEN REPUBLIK INDONESIA
        # ---
        text = re.sub(
            r"PRESIDEN\s+REPUBLIK\s+INDONESIA",
            "",
            text,
            flags=re.IGNORECASE
        )

        # ---
        # Normalize spaces
        # ---
        text = re.sub(
            r"[ \t]+",
            " ",
            text
        )

        # ---
        # Remove trailing spaces
        # ---
        lines = [line.strip() for line in text.splitlines()]

        text = "\n".join(lines)

        # ---
        # Max 2 blank lines
        # ---
        text = re.sub(
            r"\n{3,}",
            "\n\n",
            text
        )

        return text.strip()
    
    def clean_file(self, input_path: str, output_path: str) -> None:
        input_path = Path(input_path)
        output_path = Path(output_path)

        logger.info(f"Cleaning: {input_path.name}")

        text = input_path.read_text(encoding="utf-8")

        cleaned_text = self.clean(text)

        output_path.parent.mkdir(parents=True, exist_ok=True)

        output_path.write_text(cleaned_text, encoding="utf-8")

        logger.success(f"Saved: {output_path}")

In [5]:
if __name__ == "__main__":
    cleaner = TextCleaner()

    cleaner.clean_file(
        input_path="regulation/extracted/uu/uu_32_2009_pplh.txt",
        output_path="regulation/cleaned/uu/uu_32_2009_pplh_clean.txt"
    )

2026-06-02 14:23:35.183 | INFO     | __main__:clean_file:80 - Cleaning: uu_32_2009_pplh.txt
2026-06-02 14:23:35.230 | SUCCESS  | __main__:clean_file:90 - Saved: regulation\cleaned\uu\uu_32_2009_pplh_clean.txt


# Test Parser

In [2]:
from pathlib import Path
from parser.legal_parser import LegalParser

In [3]:
cleaned_text = Path(
	"regulation/cleaned/uu/uu_32_2009_pplh_clean.txt"
).read_text(encoding="utf-8")

parser = LegalParser(
	document_type="UU",
	document_number="32",
	document_year=2009,
	document_title="Perlindungan dan Pengelolaan Lingkungan Hidup"
)

result = parser.parse(cleaned_text)

print(len(result["articles"]))

print(result["general_explanation"][:500])

print(result["article_explanation"][:500])

2026-06-02 19:48:36.382 | INFO     | parser.legal_parser:parse:223 - Parsed 144 articles
2026-06-02 19:48:36.387 | INFO     | parser.legal_parser:parse:224 - General explanation found: True
2026-06-02 19:48:36.389 | INFO     | parser.legal_parser:parse:225 - Article explanation found: True


144
PENJELASAN
ATAS
UNDANG-UNDANG REPUBLIK INDONESIA
NOMOR 32 TAHUN 2009
TENTANG
PERLINDUNGAN DAN PENGELOLAAN LINGKUNGAN HIDUP
I.
UMUM
1.
Undang-Undang Dasar Negara Republik Indonesia Tahun 1945
menyatakan
bahwa
lingkungan
hidup
yang
baik
dan
sehat
merupakan hak asasi dan hak konstitusional bagi setiap warga
negara
Indonesia.
Oleh
karena
itu,
negara,
pemerintah,
dan
seluruh pemangku kepentingan berkewajiban untuk melakukan
perlindungan
dan
pengelolaan
lingkungan
hidup
dalam
pelaksanaan pembangunan b
II. PASAL DEMI PASAL
Pasal 1
Cukup jelas.
Pasal 2
Huruf a
Yang dimaksud dengan “asas tanggung jawab negara” adalah:
a.
negara
menjamin
pemanfaatan
sumber
daya
alam
akan
memberikan manfaat yang sebesar-besarnya bagi kesejahteraan
dan
mutu
hidup
rakyat,
baik
generasi
masa
kini
maupun
generasi masa depan.
b.
negara menjamin hak warga negara atas lingkungan hidup yang
baik dan sehat.
c.
negara mencegah dilakukannya kegiatan pemanfaatan sumber
daya alam yang menimbulkan pencemaran dan/atau kerus

In [ ]:
# Cek jumlah bab yang terdeteksi
result_split = parser.split_document(cleaned_text)

chapters = parser.extract_chapters(result_split["body"])

print(len(chapters))

for ch in chapters:
    print(
        ch["number"],
        ch["title"][:50]
    )

17
I KETENTUAN UMUM
II ASAS, TUJUAN, DAN RUANG LINGKUP
III PERENCANAAN
IV PEMANFAATAN
V PENGENDALIAN
VI PEMELIHARAAN
VII PENGELOLAAN BAHAN BERBAHAYA DAN BERACUN
VIII SISTEM INFORMASI
IX TUGAS DAN WEWENANG PEMERINTAH DAN PEMERINTAH DAERA
X HAK, KEWAJIBAN, DAN LARANGAN
XI PERAN MASYARAKAT
XII PENGAWASAN DAN SANKSI ADMINISTRATIF
XIII PENYELESAIAN SENGKETA LINGKUNGAN
XIV PENYIDIKAN DAN PEMBUKTIAN
XV KETENTUAN PIDANA
XVI KETENTUAN PERALIHAN
XVII KETENTUAN PENUTUP


In [ ]:
# Cek apakah ada pasal duplikat
numbers = [
    a.article_number
    for a in result["articles"]
]

from collections import Counter

counter = Counter(numbers)

duplicates = {
    k:v
    for k,v in counter.items()
    if v > 1
}

print(len(duplicates))
print(duplicates)

14
{'5': 2, '13': 2, '23': 2, '26': 2, '29': 2, '31': 2, '32': 2, '36': 3, '40': 2, '48': 3, '49': 2, '58': 2, '71': 2, '119': 3}


In [ ]:
# Cek pola pasal yang duplikat
for match in parser.ARTICLE_PATTERN.finditer(result_split["body"]):

    if match.group(1) == "36":

        print("=" * 100)

        start = match.start()

        end = min(
            len(result_split["body"]),
            start + 100
        )

        print(result_split["body"][start:end])

Pasal 36
(1) Setiap usaha dan/atau kegiatan yang wajib
memiliki
amdal
atau
UKL-UPL
wajib
memiliki iz
Pasal
36
sampai
dengan
Pasal
40
diatur
dalam
Peraturan Pemerintah.
Paragraf 8
Instrumen Ekonomi Ling
Pasal
36
ayat
(1)
wajib
menyediakan
dana
penjaminan
untuk
pemulihan fungsi lingkungan hidup.
(2)
Dan


In [ ]:
# Cek pola regex untuk deteksi header pasal
matches = list(
    re.finditer(
        r"^Pasal[ \t]+(\d+[A-Z]?)\s*$",
        result_split["body"],
        re.MULTILINE
    )
)

print(len(matches))
print(matches[-1].group(1))

127
127
